# SQL Notebook for SQL Server 2022 data virtualization for parquet files on S3 object storage

This is a SQL notebook to show the data virtualization capabilities in SQL Server 2022 for parquet files on S3 object storage. You can find all the scripts and files referenced in this notebook at https://aka.ms/sqlserver2022demos under the datavirt/parquet folder

# Prerequisites for the exercise

- SQL Server 2022 Evaluation Edition with the Database Engine and PolyBase Query Service for External Data Feature installed. You can use the defaults in setup for Polybase.
- VM or computer with 2 CPUs and at least 8Gb RAM.
- SQL Server Management Studio (SSMS). The latest 18.x build or 19.x build will work.

    **Note**: The following pre-requisites are for non-Microsoft software. The use of this software does not represent any official endorsement from Microsoft. This software is not supported by Microsoft so any issues using this software are up to the user to resolve.

- The **minio** server for Windows which you can download at https://min.io/download#/windows. For the demo I assume you have created a directory called c:\minio and have downloaded the minio.exe for Windows into that directory.
- openssl for Windows which you can download at https://slproweb.com/products/Win32OpenSSL.html. I chose the Win64 OpenSSL v3.0.5 MSI option.

# Steps to setup minio for the exercise

- Download minio.exe for Windows into c:\minio.exe
- Download the openssl for Windows MSI and run the installer. Use all the defaults.
- Set a system environment variable OPENSSL_CONF=C:\Program Files\OpenSSL-Win64\bin\openssl.cfg and add to the system environment variable PATH c:\Program Files\OpenSSL-Win64\bin
- Generate a private key using the following command from the c:\minio directory.

    `openssl genrsa -out private.key 2048`

- Copy the supplied openssl.conf to c:\minio. Edit openssl.conf by changing IP.2 to your local IP address and DNS.2 to your local computer name.
- From the c:\minio directory run the following command to generate a self-signed certificate

    `openssl req -new -x509 -nodes -days 730 -key private.key -out public.crt -config openssl.conf`

- Double-click the public.crt file and select Install Certificate. Choose Local Machine, Place all certificates in the following store, Browse, and select Trusted Root Certification Authorities.
- Copy the private.key and public.crt files from c:\minio into %%USERPROFILE%%\.minio\certs.

# Steps to use mino for the exercise

From a command prompt at the c:\minio directory start the minio server with the following command (example syntax for Powershell)

`.\minio.exe server c:\minio --console-address ":9001"`

This program will startup and "run forever" until stopped with <Ctrl>+<C>. Your output should look similar to the following (the IP addresses will match your local IP):

```
MinIO Object Storage Server
Copyright: 2015-2022 MinIO, Inc.
License: GNU AGPLv3 <https://www.gnu.org/licenses/agpl-3.0.html>
Version: RELEASE.2022-07-30T05-21-40Z (go1.18.4 windows/amd64)

Status:         1 Online, 0 Offline.
API: https://<local IP>:9000  https://127.0.0.1:9000
RootUser: <user>
RootPass: <password>
Console: https://<local IP>:9001 https://127.0.0.1:9001
RootUser: <user>
RootPass: <password>

Command-line: https://docs.min.io/docs/minio-client-quickstart-guide
   $ mc.exe alias set myminio https://<local IP>:9000 <user> <password>
``
Documentation: https://docs.min.io
```
1. Test your connection and browse the minio storage system using a web browser. Use the address https://127.0.0.1:9001. You should be presented with a login screen. Use the defaut root user and password which are displayed on the minio server screen.

1. On the left-hand side menu, click on Identity and Users. Select Create User. Create a user name with password. Select the readwrite policy for the user. This is the user and password you will use for the SECRET value in creates3creds.sql.

3. Select menu for Buckets. Select Create Bucket. Use a Bucket Name of **wwi**. Leave all defaults and select Create Bucket.

# Steps to use SQL Server 2022 for the exercise

Run each cell one at time to use SQL Server 2022 for the exercise

## Step 1 - Get WideWorldImporters

Copy the **WideWorldImporters** sample database from [https://aka.ms/WideWorldImporters](https://aka.ms/WideWorldImporters) to a local directory (The restore script assumes **c:\\sql\_sample\_databases**)

## Step 2 - Restore WideWorldImporters

Edit the following T-SQL to restore the WideWorldImporters database for your correct files paths and execute the statement

In [4]:
USE master;
GO
DROP DATABASE IF EXISTS WideWorldImporters;
GO
RESTORE DATABASE WideWorldImporters FROM DISK = 'c:\sql_sample_databases\WideWorldImporters-Full.bak' with
MOVE 'WWI_Primary' TO 'c:\sql_sample_databases\WideWorldImporters.mdf',
MOVE 'WWI_UserData' TO 'c:\sql_sample_databases\WideWorldImporters_UserData.ndf',
MOVE 'WWI_Log' TO 'c:\sql_sample_databases\WideWorldImporters.ldf',
MOVE 'WWI_InMemory_Data_1' TO 'c:\sql_sample_databases\WideWorldImporters_InMemory_Data_1',
stats=5;
GO
ALTER DATABASE WideWorldImporters SET QUERY_STORE CLEAR ALL;
GO

Commands completed successfully.

Commands completed successfully.

5 percent processed.

10 percent processed.

15 percent processed.

20 percent processed.

... (105 outputs left out)

100 percent processed.

RESTORE DATABASE successfully processed 58455 pages in 1.801 seconds (253.567 MB/sec).

Commands completed successfully.

Total execution time: 00:00:07.326

# Step 3 - Configure Polybase

Enable polybase configuration options

In [5]:
EXEC sp_configure 'polybase enabled', 1;
GO
RECONFIGURE;
GO
EXEC sp_configure 'allow polybase export', 1;
GO
RECONFIGURE;
GO

Configuration option 'polybase enabled' changed from 1 to 1. Run the RECONFIGURE statement to install.

Commands completed successfully.

Configuration option 'allow polybase export' changed from 1 to 1. Run the RECONFIGURE statement to install.

Commands completed successfully.

Total execution time: 00:00:00.037

# Step 4 - Master Key

Create a master key to protect the database scoped credential. Edit the following statement to put in a strong password

In [7]:
USE [WideWorldImporters]
GO
IF NOT EXISTS (SELECT * FROM sys.symmetric_keys WHERE name = '##MS_DatabaseMasterKey##')
	CREATE MASTER KEY ENCRYPTION BY PASSWORD = '<password>';
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.057

# Step 5 - Database Scoped Credential

Create a database scoped credential. Edit the following statement to put in the user and password from RootUser and RootPwd from minio

In [9]:
USE [WideWorldImporters];
GO
-- Have to drop the data source first if it exists
IF EXISTS (SELECT * FROM sys.external_data_sources WHERE name = 's3_wwi')
	DROP EXTERNAL DATA SOURCE s3_wwi;
IF EXISTS (SELECT * FROM sys.database_scoped_credentials WHERE name = 's3_wwi_cred')
    DROP DATABASE SCOPED CREDENTIAL s3_wwi_cred;
GO
CREATE DATABASE SCOPED CREDENTIAL s3_wwi_cred
WITH IDENTITY = 'S3 Access Key',
SECRET = '<user>:<password>';
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.008

# Step 6 - External Data Source

Create an external data source. Edit the script to put in your local IP address

In [10]:
USE [WideWorldImporters];
GO
IF EXISTS (SELECT * FROM sys.external_data_sources WHERE name = 's3_wwi')
	DROP EXTERNAL DATA SOURCE s3_wwi;
GO
CREATE EXTERNAL DATA SOURCE s3_wwi
WITH
(
 LOCATION = 's3://<local IP>:9000'
,CREDENTIAL = s3_wwi_cred
);
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.006

# Step 7 - External file format

Create an external format file for paruqet to be used for external tables

In [11]:
USE [WideWorldImporters];
GO
IF EXISTS (SELECT * FROM sys.external_file_formats WHERE name = 'ParquetFileFormat')
	DROP EXTERNAL FILE FORMAT ParquetFileFormat;
CREATE EXTERNAL FILE FORMAT ParquetFileFormat WITH(FORMAT_TYPE = PARQUET);
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.009

# Step 8 - Use CETAS

Create a parquet file based on creation of an external table from a user table

In [17]:
USE [WideWorldImporters];
GO
IF OBJECT_ID('wwi_customer_transactions', 'U') IS NOT NULL
	DROP EXTERNAL TABLE wwi_customer_transactions;
GO
CREATE EXTERNAL TABLE wwi_customer_transactions
WITH (
    LOCATION = '/wwi/',
    DATA_SOURCE = s3_wwi,  
    FILE_FORMAT = ParquetFileFormat
) 
AS
SELECT * FROM Sales.CustomerTransactions;
GO

Commands completed successfully.

Commands completed successfully.

(97147 rows affected)

Total execution time: 00:00:00.188

# Step 9 - Browse S3

Use the minio console to browse the wwi bucket to see the parquet file created. You will se a file like `<guid>`-1-1.parquet

# Step 10 - Query external table

Run a query against the created external table to access parquet data. Join with a table in the database.

In [18]:
USE [WideWorldImporters];
GO
SELECT c.CustomerName, SUM(wct.OutstandingBalance) as total_balance
FROM wwi_customer_transactions wct
JOIN Sales.Customers c
ON wct.CustomerID = c.CustomerID
GROUP BY c.CustomerName
ORDER BY total_balance DESC;
GO

Commands completed successfully.

(263 rows affected)

Total execution time: 00:00:00.439

CustomerName,total_balance
Matyas Sedlar,0.00
Richard Friar,0.00
Eva Schulteisz,0.00
Tai Bach,0.00
Sumati Chatterjee,0.00
Victoria Lacusta,0.00
Gayatri Gajula,0.00
Isidora Morales,0.00
Pari Hosseini,0.00
Pinja Jantunen,0.00


# Step 11 - Use OPENROWSET() to query parquet files

Execute the following T-SQL to query parquet files stored in S3

In [19]:
USE [WideWorldImporters];
GO
SELECT *
FROM OPENROWSET
	(BULK '/wwi/'
	, FORMAT = 'PARQUET'
	, DATA_SOURCE = 's3_wwi')
as [wwi_customer_transactions_file];
GO

Commands completed successfully.

(97147 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:01.046

CustomerTransactionID,CustomerID,TransactionTypeID,InvoiceID,PaymentMethodID,TransactionDate,AmountExcludingTax,TaxAmount,TransactionAmount,OutstandingBalance,FinalizationDate,IsFinalized,LastEditedBy,LastEditedWhen
2,832,1,1,NULL,2013-01-01,2300.00,345.00,2645.00,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
5,803,1,2,NULL,2013-01-01,405.00,60.75,465.75,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
7,1,1,3,NULL,2013-01-01,90.00,13.50,103.50,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
11,1,1,4,NULL,2013-01-01,445.20,66.78,511.98,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
15,905,1,5,NULL,2013-01-01,704.00,105.60,809.60,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
19,976,1,6,NULL,2013-01-01,430.00,64.50,494.50,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
24,401,1,7,NULL,2013-01-01,603.50,90.53,694.03,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
28,964,1,8,NULL,2013-01-01,258.00,38.70,296.70,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
31,1,1,9,NULL,2013-01-01,178.00,26.70,204.70,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000
34,1,1,10,NULL,2013-01-01,104.00,15.60,119.60,0.00,2013-01-02,1,10,2013-01-02 11:30:00.0000000


# Step 12 - Create an external table

Execute the following T-SQL to create an external table based on a parquet file with a subset of columns

In [20]:
USE [WideWorldImporters];
GO
IF OBJECT_ID('wwi_customer_transactions_base', 'U') IS NOT NULL
	DROP EXTERNAL TABLE wwi_customer_transactions_base;
GO
CREATE EXTERNAL TABLE wwi_customer_transactions_base 
( 
	CustomerTransactionID int, 
	CustomerID int,
	TransactionTypeID int,
	TransactionDate date,
	TransactionAmount decimal(18,2)
)
WITH 
(
	LOCATION = '/wwi/'
    , FILE_FORMAT = ParquetFileFormat
    , DATA_SOURCE = s3_wwi
);
GO
SELECT * FROM wwi_customer_transactions_base;
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

(97147 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.453

CustomerTransactionID,CustomerID,TransactionTypeID,TransactionDate,TransactionAmount
9183,401,3,2013-02-08,-35321.10
9184,810,3,2013-02-08,-2885.35
9185,821,3,2013-02-08,-6520.50
9186,822,3,2013-02-08,-616.40
9187,839,3,2013-02-08,-7423.25
9188,851,3,2013-02-08,-1449.00
9189,873,3,2013-02-08,-2178.10
9190,878,3,2013-02-08,-2663.40
9191,882,3,2013-02-08,-1607.70
9192,920,3,2013-02-08,-3220.00


# Step 13 - Create statistics on an external table

Execute the following T-SQL to create statistics on an external table

In [21]:
USE [WideWorldImporters];
GO
CREATE STATISTICS wwi_ctb_stats ON wwi_customer_transactions_base (CustomerID) WITH FULLSCAN;
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.383

# Step 14 - Explorer data virtualization metadata

Execute the following T-SQL statements to explore data virtualization metadata

In [22]:
USE [WideWorldImporters];
GO
SELECT * FROM sys.external_data_sources;
GO
SELECT * FROM sys.external_file_formats;
GO
SELECT * FROM sys.external_tables;
GO

Commands completed successfully.

(1 row affected)

(1 row affected)

(2 rows affected)

Total execution time: 00:00:00.094

data_source_id,name,location,type_desc,type,resource_manager_location,credential_id,database_name,shard_map_name,connection_options,pushdown
65536,s3_wwi,s3://192.168.232.131:9000,NONE,6,NULL,65537,NULL,NULL,NULL,ON


file_format_id,name,format_type,field_terminator,string_delimiter,date_format,use_type_default,serde_method,row_terminator,encoding,data_compression,first_row,parser_version
65536,ParquetFileFormat,PARQUET,NULL,,,0,NULL,NULL,NULL,NULL,NULL,NULL


name,object_id,principal_id,schema_id,parent_object_id,type,type_desc,create_date,modify_date,is_ms_shipped,is_published,is_schema_published,max_column_id_used,uses_ansi_nulls,data_source_id,file_format_id,location,reject_type,reject_value,reject_sample_value,distribution_type,distribution_desc,sharding_col_id,remote_schema_name,remote_object_name,rejected_row_location,table_options,partition_type,partition_desc
wwi_customer_transactions,1899153811,NULL,1,0,U,USER_TABLE,2022-08-03 19:58:44.430,2022-08-03 19:58:44.430,0,0,0,14,1,65536,65536,/wwi/,VALUE,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,NULL
wwi_customer_transactions_base,1915153868,NULL,1,0,U,USER_TABLE,2022-08-03 20:32:12.967,2022-08-03 20:32:12.967,0,0,0,5,1,65536,65536,/wwi/,VALUE,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,NULL


# Step 15 - Explore parquet file metadata

Execute the following T-SQL statement to explore metadata about columns in the parquet file

In [23]:
USE [WideWorldImporters];
GO
EXEC sp_describe_first_result_set N'
SELECT *
FROM OPENROWSET
	(BULK ''/wwi/''
	, FORMAT = ''PARQUET''
	, DATA_SOURCE = ''s3_wwi'')
as [wwi_customer_transactions_file];';
GO

Commands completed successfully.

(14 rows affected)

Total execution time: 00:00:00.039

is_hidden,column_ordinal,name,is_nullable,system_type_id,system_type_name,max_length,precision,scale,collation_name,user_type_id,user_type_database,user_type_schema,user_type_name,assembly_qualified_type_name,xml_collection_id,xml_collection_database,xml_collection_schema,xml_collection_name,is_xml_document,is_case_sensitive,is_fixed_length_clr_type,source_server,source_database,source_schema,source_table,source_column,is_identity_column,is_part_of_unique_key,is_updateable,is_computed_column,is_sparse_column_set,ordinal_in_order_by_list,order_by_is_descending,order_by_list_length,tds_type_id,tds_length,tds_collation_id,tds_collation_sort_id
0,1,CustomerTransactionID,1,56,int,4,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,38,4,NULL,NULL
0,2,CustomerID,1,56,int,4,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,38,4,NULL,NULL
0,3,TransactionTypeID,1,56,int,4,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,38,4,NULL,NULL
0,4,InvoiceID,1,56,int,4,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,38,4,NULL,NULL
0,5,PaymentMethodID,1,56,int,4,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,38,4,NULL,NULL
0,6,TransactionDate,1,40,date,3,10,0,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,40,3,NULL,NULL
0,7,AmountExcludingTax,1,108,"numeric(18,2)",9,18,2,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,108,17,NULL,NULL
0,8,TaxAmount,1,108,"numeric(18,2)",9,18,2,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,108,17,NULL,NULL
0,9,TransactionAmount,1,108,"numeric(18,2)",9,18,2,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,108,17,NULL,NULL
0,10,OutstandingBalance,1,108,"numeric(18,2)",9,18,2,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,0,0,NULL,NULL,NULL,NULL,NULL,0,NULL,1,0,0,NULL,NULL,NULL,108,17,NULL,NULL


# Step 16 - Explore file metadata

Execute the following T-SQL statement to explore metadata from the external table about the source parquet files

In [24]:
USE [WideWorldImporters];
GO
SELECT TOP 1 wwi_customer_transactions_file.filepath(), 
wwi_customer_transactions_file.filename()
FROM OPENROWSET
	(BULK '/wwi/'
	, FORMAT = 'PARQUET'
	, DATA_SOURCE = 's3_wwi')
as [wwi_customer_transactions_file];
GO

Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.033

(No column name),(No column name)
s3://192.168.232.131:9000/wwi/4E605C68-A1C5-4C42-B1E8-B935C448347F-1-1.parquet,4E605C68-A1C5-4C42-B1E8-B935C448347F-1-1.parquet
